# Chunking as a Segmentation and Optimization Problem — companion notebook

The **narrative** half of the `chunking-as-segmentation` topic. One source of truth:
`chunking_as_segmentation.py` owns every number; this notebook imports it and walks the topic
section by section, running each assert so the claims render as executed output.

Run from the repo root:
```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/chunking-as-segmentation/01_chunking_as_segmentation.ipynb
```

In [ ]:
import sys, pathlib
_slug, _mod = "chunking-as-segmentation", "chunking_as_segmentation.py"
_cands = [pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks" / _slug]
_cands += [p / "notebooks" / _slug for p in pathlib.Path.cwd().parents]
for _c in _cands:
    if (_c / _mod).exists():
        sys.path.insert(0, str(_c)); break

import numpy as np
from chunking_as_segmentation import (
    K_GRID, synthetic_document, segment_cost, total_coherence_cost, mean_resultant_length,
    segment_dp, brute_force_segment, adjacent_dissimilarity, texttiling_greedy, fixed_size,
    boundary_f1, grid_table, finance_demo,
    test_coherence_is_resultant_length, test_dp_matches_brute_force, test_dp_beats_heuristics,
    test_cost_monotone_in_k, test_dp_recovers_boundaries, test_finance_filing,
)
print("imported chunking_as_segmentation")

## 1. Coherence is the mean resultant length

Chunking asks where to cut a sequence of sentence embeddings. Cast it as maximizing within-chunk
coherence. For L2-normalized embeddings $e_t$, the incoherence of a segment $[i,j)$ is
$$ c(i,j) = (j-i) - \Big\lVert \sum_{t\in[i,j)} e_t \Big\rVert = (j-i)\,(1-\bar R), $$
where $\bar R = \lVert\text{mean}(e_t)\rVert$ is the **mean resultant length** — the von Mises–Fisher
concentration statistic from the prerequisite topic. Minimizing total cost carves the document into
tight clusters on the sphere.

In [ ]:
test_coherence_is_resultant_length()

## 2. The optimal segmentation is a dynamic program

Because the cost is additive across segments, the globally optimal segmentation into $k$ contiguous
chunks is computed exactly by dynamic programming,
$$ D[s][j] = \min_{i<j} \, D[s-1][i] + c(i,j), $$
in $O(k n^2)$. The check confirms it matches an exhaustive brute-force search at every $k$ — it is
the global optimum, not a heuristic.

In [ ]:
test_dp_matches_brute_force()

## 3. Heuristics cannot beat the optimum

TextTiling places boundaries greedily at the gaps of highest adjacent dissimilarity; fixed-size
chunking ignores meaning entirely. Neither can beat the DP's objective value at the same number of
segments — the DP is optimal by construction.

In [ ]:
test_dp_beats_heuristics()

## 4. More segments always lower the cost

The optimal within-segment cost is monotone nonincreasing in the number of segments — splitting more
can only reduce within-chunk incoherence. This is the granularity knob, and the warning that comes
with it.

In [ ]:
test_cost_monotone_in_k()

## 5. Coherence is a proxy

On a document with planted topic shifts, the DP recovers the true boundaries better than the
heuristics. But watch the next section's table: boundary-recovery F1 *peaks at the true segment
count* and then *declines* under over-segmentation, even as the coherence cost keeps falling.
Optimizing coherence is not the same as optimizing the structure we actually want — coherence is a
proxy for downstream retrieval quality.

In [ ]:
test_dp_recovers_boundaries()

## 6. The grid table and the finance filing

`grid_table()` is the block `ChunkingLaboratory.tsx` mirrors; `finance_demo()` is the synthetic 10-K
headline. Notice the DP cost falling monotonically with $k$ while the DP F1 humps at the true
$k = 5$ — the proxy mismatch — and the DP recovering the section structure the baselines miss.

In [ ]:
tbl = grid_table()
print(f"document: {tbl['n']} sentences, planted boundaries {tbl['truth']}, fixed-size F1 {tbl['fixed_f1']}")
print(f"{'k':>4}{'dp_cost':>10}{'dp_F1':>8}{'greedy_cost':>13}{'greedy_F1':>11}")
for r in tbl['perK']:
    print(f"{r['k']:>4}{r['dp_cost']:>10.3f}{r['dp_f1']:>8.2f}{r['greedy_cost']:>13.3f}{r['greedy_f1']:>11.2f}")

In [ ]:
test_finance_filing()

## What we verified

Every claim ran as an assert: coherence equals the mean resultant length, the DP matches brute-force
optimality, the heuristics cannot beat it, the cost is monotone in the segment count, the DP recovers
planted boundaries better than the baselines, and the finance filing. The grid above — cost falling
while F1 humps — is the honest reminder that the optimized objective is a proxy.

In [ ]:
print('All chunking-as-segmentation claims verified against chunking_as_segmentation.py')